# Sistemi lineari 


Consideriamo un sistema lineare quadrato

$$
A\mathbf{x}=\mathbf{b},
\qquad
A\in\mathbb{R}^{n\times n},
\quad\mathbf{x},\mathbf{b}\in\mathbb{R}^{n}.
$$

Il vettore $\mathbf{b}$ è noto, mentre $\mathbf{x}$ è il vettore delle incognite.

Lo studio numerico dei sistemi lineari comprende tre questioni fondamentali:

1. **Esistenza e unicità:** il sistema possiede una soluzione? La soluzione è unica?
2. **Algoritmi:** come possiamo calcolare la soluzione in modo efficiente?
3. **Errore:** quanto è affidabile la soluzione calcolata in aritmetica finita?

## Esistenza e unicità della soluzione

Se $A$ è quadrata, le seguenti proprietà sono equivalenti:

- $A$ è non singolare;
- $\det(A)\neq0$;
- $\operatorname{rank}(A)=n$;
- $\ker(A)=\{\mathbf{0}\}$;
- esiste $A^{-1}$;
- per ogni $\mathbf{b}\in\mathbb{R}^n$, il sistema $A\mathbf{x}=\mathbf{b}$ ammette un'unica soluzione.

In questo caso, dal punto di vista teorico,

$$
\mathbf{x}=A^{-1}\mathbf{b}.
$$

Se $A$ è singolare, il sistema può non avere soluzioni oppure può averne infinite, a seconda del vettore $\mathbf{b}$.

## Perché non si calcola l'inversa?

La formula $\mathbf{x}=A^{-1}\mathbf{b}$ è utile dal punto di vista teorico, ma normalmente **non viene utilizzata come algoritmo**.

Calcolare esplicitamente $A^{-1}$:

- richiede più operazioni della soluzione diretta del sistema;
- richiede memoria per memorizzare l'intera matrice inversa;
- può introdurre errori di arrotondamento non necessari;
- calcola molte informazioni che non servono se desideriamo soltanto $\mathbf{x}$.

Nella pratica si usa quindi un algoritmo che risolve direttamente

$$
A\mathbf{x}=\mathbf{b}
$$

senza costruire $A^{-1}$.

## Metodi diretti e metodi iterativi

```{figure} immagini_sorgente/metodi_diretti_iterativi.png
---
width: 100%
align: center
---
```
<p align="center">
  <img src="immagini_sorgente/metodi_diretti_iterativi.png" width="700">
</p>

I **metodi diretti** producono la soluzione dopo un numero finito di operazioni, se si lavora in aritmetica esatta. Sono basati principalmente sull'eliminazione di Gauss e sulle fattorizzazioni della matrice.

I **metodi iterativi** costruiscono invece una successione di approssimazioni

$$
\mathbf{x}^{(0)},\mathbf{x}^{(1)},\mathbf{x}^{(2)},\ldots
$$

che, sotto opportune condizioni, converge alla soluzione. Sono particolarmente importanti per matrici grandi e sparse.

## Matrici particolari: sistemi triangolari

Una matrice $L=(\ell_{ij})$ è **triangolare inferiore** se

$$
\ell_{ij}=0\qquad\text{per }j>i.
$$

Una matrice $U=(u_{ij})$ è **triangolare superiore** se

$$
u_{ij}=0\qquad\text{per }i>j.
$$

I sistemi triangolari sono semplici da risolvere perché le incognite possono essere calcolate una alla volta.



Consideriamo il sistema triangolare inferiore

$$
L\mathbf{x}=\mathbf{b}.
$$

La prima equazione contiene soltanto $x_1$. Una volta calcolato $x_1$, la seconda equazione permette di determinare $x_2$, e così via.

La formula generale è

$$
x_i=\frac{1}{\ell_{ii}}
\left(b_i-\sum_{j=1}^{i-1}\ell_{ij}x_j\right),
\qquad i=1,\ldots,n.
$$

Il metodo prende il nome di **sostituzione in avanti**. Richiede un numero di operazioni proporzionale a $n^2$.


Consideriamo ora il sistema triangolare superiore

$$
U\mathbf{x}=\mathbf{b}.
$$

Si parte dall'ultima equazione e si procede verso la prima:

$$
x_i=\frac{1}{u_{ii}}
\left(b_i-\sum_{j=i+1}^{n}u_{ij}x_j\right),
\qquad i=n,n-1,\ldots,1.
$$

Questo procedimento è detto **sostituzione all'indietro** e ha anch'esso complessità $\mathcal{O}(n^2)$.

In [22]:
import numpy as np

np.set_printoptions(precision=5, suppress=True)

def sostituzione_avanti(L, b):
    L = np.asarray(L, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(b)
    x = np.zeros(n)

    for i in range(n):
        if abs(L[i, i]) < 1e-14:
            raise ValueError('Elemento diagonale nullo')
        x[i] = (b[i] - L[i, :i] @ x[:i]) / L[i, i]
    return x


def sostituzione_indietro(U, b):
    U = np.asarray(U, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(b)
    x = np.zeros(n)

    for i in range(n - 1, -1, -1):
        if abs(U[i, i]) < 1e-14:
            raise ValueError('Elemento diagonale nullo')
        x[i] = (b[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
    return x

## Costo computazionale

Per un sistema triangolare di ordine $n$, il numero complessivo di termini presenti nelle somme è

$$
1+2+\cdots+(n-1)=\frac{n(n-1)}{2}.
$$

Una sostituzione in avanti o all'indietro richiede quindi circa $n^2$ operazioni aritmetiche, cioè

$$
\mathcal{O}(n^2).
$$

Questo è molto meno del costo $\mathcal{O}(n^3)$ necessario per fattorizzare una matrice densa generale.

# Risolvere un sistema mediante fattorizzazione LU

Consideriamo ora il sistema 

$$
A\mathbf{x}=\mathbf{b}
$$


```{figure} immagini_sorgente/risoluzione_con_lu.png
---
width: 100%
align: center
---
```
<p align="center">
  <img src="immagini_sorgente/risoluzione_con_lu.png" width="700">
</p>

Infatti, il sistema $A\mathbf{x}=\mathbf{b}$ diventa

$$
LU\mathbf{x}=\mathbf{b}.
$$

Ponendo $\mathbf{y}=U\mathbf{x}$, risolviamo in successione

$$
L\mathbf{y}=\mathbf{b},
\qquad
U\mathbf{x}=\mathbf{y}.
$$

### Esempio
Consideriamo ora il sistema 

$$
A\mathbf{x}=\mathbf{b}
$$

in cui la matrice $A$ è la precedente:

$$
A=
\begin{pmatrix}
2&1&1\\
4&-6&0\\
-2&7&2
\end{pmatrix}.
$$
 e il termine noto è: 

$$
\mathbf{b}=
\begin{pmatrix}
5\\-2\\9
\end{pmatrix}.
$$

Dalla fattorizzazione $A=LU$ abbiamo ottenuto le seguenti matrici:

$$
U=
\begin{pmatrix}
2&1&1\\
0&-8&-2\\
0&0&1
\end{pmatrix}.
$$

$$
L=
\begin{pmatrix}
1&0&0\\
2&1&0\\
-1&-1&1
\end{pmatrix}.
$$

### Primo passaggio: $L\mathbf{y}=\mathbf{b}$

La sostituzione in avanti fornisce

$$
\mathbf{y}=
\begin{pmatrix}
5\\-12\\2
\end{pmatrix}.
$$

### Secondo passaggio: $U\mathbf{x}=\mathbf{y}$

La sostituzione all'indietro fornisce

$$
\mathbf{x}=
\begin{pmatrix}
1\\1\\2
\end{pmatrix}.
$$

In [23]:
A = np.array([[ 2.,  1., 1.],
              [ 4., -6., 0.],
              [-2.,  7., 2.]])

L = np.array([[ 1.,  0., 0.],
              [ 2.,  1., 0.],
              [-1., -1., 1.]])

U = np.array([[2.,  1.,  1.],
              [0., -8., -2.],
              [0.,  0.,  1.]])

print('L @ U =\n', L @ U)
print('\nA = L @ U?', np.allclose(A, L @ U))
print('Errore di fattorizzazione:', np.linalg.norm(A - L @ U))

L @ U =
 [[ 2.  1.  1.]
 [ 4. -6.  0.]
 [-2.  7.  2.]]

A = L @ U? True
Errore di fattorizzazione: 0.0


In [24]:
b = np.array([5., -2., 9.])

y = sostituzione_avanti(L, b)
x = sostituzione_indietro(U, y)

print('y =', y)
print('x =', x)
print('Residuo b - Ax =', b - A @ x)
print('Norma del residuo =', np.linalg.norm(b - A @ x))

y = [  5. -12.   2.]
x = [1. 1. 2.]
Residuo b - Ax = [0. 0. 0.]
Norma del residuo = 0.0


### Risoluzione di un sistema con fattorizzazione Lu con pivoting

Quando la matrice $A$ è fattorizzata con pivoting, si ha che:

$PA=LU$ dove $P$ è una matrice di permutazione.

Allora il sistema lineare $Ax=b$ è equivalente al sistema: $PAx=Pb$, cioè $LUx=Pb$.

Si risolvono quindi in sequenza:
1. $Ly=Pb$
2. $Ux=y$.




Scegliamo

$$
\mathbf{b}=
\begin{pmatrix}
3\\0\\7
\end{pmatrix}.
$$

Poiché $PA=LU$, dobbiamo prima permutare il termine noto:

$$
P\mathbf{b}=
\begin{pmatrix}
7\\0\\3
\end{pmatrix}.
$$

La sostituzione in avanti nel sistema

$$
L\mathbf{y}=P\mathbf{b}
$$

fornisce

$$
\mathbf{y}=
\begin{pmatrix}
7\\-\frac72\\1
\end{pmatrix}.
$$

Risolvendo poi $U\mathbf{x}=\mathbf{y}$ otteniamo

$$
\mathbf{x}=
\begin{pmatrix}
1\\2\\-1
\end{pmatrix}.
$$

In [25]:
A_piv = np.array([[0.,  2.,  1.],
                  [1., -2., -3.],
                  [2.,  3.,  1.]])

P = np.array([[0., 0., 1.],
              [0., 1., 0.],
              [1., 0., 0.]])

L_piv = np.array([[1.,     0., 0.],
                  [1/2,    1., 0.],
                  [0.,   -4/7, 1.]])

U_piv = np.array([[2.,  3.,    1.],
                  [0., -7/2, -7/2],
                  [0.,  0.,   -1.]])

print('P @ A =\n', P @ A_piv)
print('\nL @ U =\n', L_piv @ U_piv)
print('\nP @ A = L @ U?', np.allclose(P @ A_piv, L_piv @ U_piv))
print('Errore di fattorizzazione:', np.linalg.norm(P @ A_piv - L_piv @ U_piv))

P @ A =
 [[ 2.  3.  1.]
 [ 1. -2. -3.]
 [ 0.  2.  1.]]

L @ U =
 [[ 2.  3.  1.]
 [ 1. -2. -3.]
 [ 0.  2.  1.]]

P @ A = L @ U? True
Errore di fattorizzazione: 0.0


In [26]:
b_piv = np.array([3., 0., 7.])

y_piv = sostituzione_avanti(L, P @ b_piv)
x_piv = sostituzione_indietro(U, y_piv)

print('P @ b =', P @ b_piv)
print('y =', y_piv)
print('x =', x_piv)
print('Residuo b - Ax =', b_piv - A_piv @ x_piv)
print('Norma del residuo =', np.linalg.norm(b_piv - A_piv @ x_piv))

P @ b = [7. 0. 3.]
y = [  7. -14.  -4.]
x = [ 4.125  2.75  -4.   ]
Residuo b - Ax = [  1.5   -10.625  -5.5  ]
Norma del residuo = 12.057803489856683


## Risoluzione di un sistema con fattorizzazione di Cholesky 

Se $A$ è una matrice simmetrica e definita positiva, abbiamo visti che si può fattorizzare tramite l'algoritmo di Cholesky come: $A=LL^T$.

Allora  il sistema

$$
A\mathbf{x}=\mathbf{b}
$$

diventa

$$
LL^T\mathbf{x}=\mathbf{b}.
$$

Si risolvono quindi due sistemi triangolari:

$$
L\mathbf{y}=\mathbf{b},
\qquad
L^T\mathbf{x}=\mathbf{y}.
$$



## Esercizi

1. Verificare manualmente che le matrici $L$ e $U$ dell'esempio soddisfino $LU=A$.
2. Risolvere lo stesso sistema con `np.linalg.solve` e confrontare il risultato.
3. Cambiare il termine noto $\mathbf{b}$ senza ricalcolare $L$ e $U$.
4. Contare moltiplicazioni e divisioni effettuate dalle due funzioni di sostituzione.
5. Provare la funzione `fattorizzazione_lu` sulla matrice

   $$
   \begin{pmatrix}
   0&1\\1&1
   \end{pmatrix}
   $$

   e spiegare perché è necessario uno scambio di righe.

# Condizionamento dei sistemi lineari

Quando risolviamo un sistema lineare $Ax=b$, i dati $A$ e $b$ sono spesso affetti da errori di misura, arrotondamento o discretizzazione. Il **condizionamento** descrive quanto la soluzione è sensibile a queste perturbazioni.

> Il condizionamento è una proprietà del **problema**. La stabilità, invece, è una proprietà dell'**algoritmo** usato per risolverlo.

## Il problema perturbato

Indichiamo con $x$ la soluzione esatta e con $x+\delta x$ la soluzione ottenuta dopo avere perturbato i dati:

$$
Ax=b, \qquad (A+\Delta A)(x+\delta x)=b+\delta b.
$$

Confrontiamo perturbazioni ed errori in forma relativa:

$$
\frac{\|\Delta A\|}{\|A\|}, \qquad
\frac{\|\delta b\|}{\|b\|}, \qquad
\frac{\|\delta x\|}{\|x\|}.
$$

Il problema è **ben condizionato** se piccole variazioni relative nei dati producono piccole variazioni relative nella soluzione; è **mal condizionato** quando gli errori possono essere fortemente amplificati.

## Un esempio mal condizionato

Consideriamo

$$
A=\begin{bmatrix}1 & 2\\0.499 & 1.001\end{bmatrix},
\qquad b=\begin{bmatrix}3\\1.5\end{bmatrix}.
$$

La soluzione è $x=(1,1)^T$. Modifichiamo soltanto due elementi della seconda riga:

$$
\widetilde A=\begin{bmatrix}1 & 2\\0.500 & 1.002\end{bmatrix}.
$$

La perturbazione relativa della matrice è circa $5.7\cdot10^{-4}$, ma la nuova soluzione è $\widetilde x=(3,0)^T$: il cambiamento relativo della soluzione è circa $1.58$.

In [27]:
import numpy as np

A = np.array([[1.000, 2.000],
              [0.499, 1.001]])
A_pert = np.array([[1.000, 2.000],
                   [0.500, 1.002]])
b = np.array([3.000, 1.500])

x = np.linalg.solve(A, b)
x_pert = np.linalg.solve(A_pert, b)
errore_A = np.linalg.norm(A_pert-A, 2) / np.linalg.norm(A, 2)
errore_x = np.linalg.norm(x_pert-x, 2) / np.linalg.norm(x, 2)

print(f'Soluzione originale:  {x}')
print(f'Soluzione perturbata: {x_pert}')
print(f'Perturbazione relativa di A: {errore_A:.3e}')
print(f'Cambiamento relativo di x:   {errore_x:.3e}')

Soluzione originale:  [1. 1.]
Soluzione perturbata: [3. 0.]
Perturbazione relativa di A: 5.656e-04
Cambiamento relativo di x:   1.581e+00


## Numero di condizionamento

Se $A$ è invertibile, rispetto a una norma matriciale indotta si definisce

$$
\kappa(A)=\|A\|\,\|A^{-1}\|.
$$

Vale sempre $\kappa(A)\geq 1$. Se viene perturbato soltanto il termine noto, allora

$$
\frac{\|\delta x\|}{\|x\|}
\leq \kappa(A)\frac{\|\delta b\|}{\|b\|}.
$$

Quindi $\kappa(A)$ è un limite superiore al fattore di amplificazione dell'errore relativo. Se sono perturbati sia $A$ sia $b$, per perturbazioni sufficientemente piccole si ha, al primo ordine,

$$
\frac{\|\delta x\|}{\|x\|}
\lesssim \kappa(A)\left(
\frac{\|\Delta A\|}{\|A\|}+
\frac{\|\delta b\|}{\|b\|}
\right).
$$

## Condizionamento nella norma 2

Se $\sigma_{\max}$ e $\sigma_{\min}$ sono il massimo e il minimo valore singolare di $A$, allora

$$
\kappa_2(A)=\|A\|_2\|A^{-1}\|_2
=\frac{\sigma_{\max}(A)}{\sigma_{\min}(A)}.
$$

Quando $\sigma_{\min}$ è molto piccolo, la matrice è vicina a essere singolare e $\kappa_2(A)$ è grande. Geometricamente, $A$ trasforma la sfera unitaria in un'ellisse: un'ellisse molto schiacciata segnala un forte condizionamento.

In [28]:
A_ben = np.array([[2.0, -1.0],
                  [-1.0, 2.0]])
A_mal = np.array([[1.000, 2.000],
                  [0.499, 1.001]])

for nome, M in [('ben condizionata', A_ben), ('mal condizionata', A_mal)]:
    x = np.ones(2)
    b = M @ x
    U, s, VT = np.linalg.svd(M)
    # Perturbazione relativa 10^-6 nella direzione più sfavorevole
    db = 1e-6 * np.linalg.norm(b) * U[:, -1]
    x_pert = np.linalg.solve(M, b + db)
    err_b = np.linalg.norm(db) / np.linalg.norm(b)
    err_x = np.linalg.norm(x_pert-x) / np.linalg.norm(x)
    print(f'{nome:18s}: kappa_2 = {np.linalg.cond(M):9.2f}, '
          f'err_rel(b) = {err_b:.1e}, err_rel(x) = {err_x:.2e}')

ben condizionata  : kappa_2 =      3.00, err_rel(b) = 1.0e-06, err_rel(x) = 1.00e-06
mal condizionata  : kappa_2 =   2083.67, err_rel(b) = 1.0e-06, err_rel(x) = 1.98e-03


## La matrice di Hilbert

Un esempio classico di matrice mal condizionata è la matrice di Hilbert:

$$
H_{ij}=\frac{1}{i+j-1}, \qquad i,j=1,\ldots,n.
$$

Il suo numero di condizionamento cresce molto rapidamente con la dimensione.

In [29]:
def hilbert(n):
    i, j = np.indices((n, n))
    return 1.0 / (i + j + 1)

for n in range(2, 11):
    print(f'n = {n:2d}   kappa_2(H) = {np.linalg.cond(hilbert(n)):10.3e}')

n =  2   kappa_2(H) =  1.928e+01
n =  3   kappa_2(H) =  5.241e+02
n =  4   kappa_2(H) =  1.551e+04
n =  5   kappa_2(H) =  4.766e+05
n =  6   kappa_2(H) =  1.495e+07
n =  7   kappa_2(H) =  4.754e+08
n =  8   kappa_2(H) =  1.526e+10
n =  9   kappa_2(H) =  4.932e+11
n = 10   kappa_2(H) =  1.603e+13


## Osservazioni pratiche

- Un valore di $\kappa(A)$ vicino a $1$ indica un problema ben condizionato; un valore molto grande indica potenziale amplificazione degli errori.
- Il valore dipende dalla norma: NumPy calcola `np.linalg.cond(A, p)`; senza specificare `p` usa la norma 2.
- In aritmetica floating point, una stima orientativa delle cifre decimali perdute è $\log_{10}\kappa(A)$.
- Un residuo piccolo $\|b-A\widehat x\|$ non garantisce necessariamente un errore piccolo $\|x-\widehat x\|$ quando il sistema è mal condizionato.
- Un algoritmo stabile non può eliminare la sensibilità intrinseca di un problema mal condizionato.